In [1]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False  # noqa: F821
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True  # noqa: F821

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

TAAF RUN_AS_SUBMISSION=False
taaf.kaggle: LIBRARY_PATH=/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs


In [2]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")  # noqa: F821
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
DATASET_SOURCES: list[str] = ["jeroencottaar/taaf-kaggle-source", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]  # noqa: F821
KERNEL_SOURCES: list[str] = []  # noqa: F821
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"  # noqa: F821
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()  # noqa: F821
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0  # noqa: F821
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

TAAF source bundle: /kaggle/input/datasets/jeroencottaar/taaf-kaggle-source-share
taaf.kaggle: input paths = {"driessmit1/arc3-vllm-h100-wheelhouse-v3": "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": "/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot", "jeroencottaar/taaf-kaggle-source": "/kaggle/input/datasets/jeroencottaar/taaf-kaggle-source-share"}


In [4]:
# Audit attached datasets
import subprocess

subprocess.run(["ls", "/kaggle/input"], check=False)

competitions
datasets


CompletedProcess(args=['ls', '/kaggle/input'], returncode=0)

In [5]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return
    commands = json.loads(path.read_text(encoding="utf-8"))
    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(str(command), shell=True, check=check, cwd=WORKING_DIR, env=env)
        if not check and result.returncode != 0:
            print(f"taaf.kaggle: {label} command exited with {result.returncode}", flush=True)
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Model setup is required only for a full validation or competition rerun.
# A normal Kaggle save exists solely to create the code-submission artifact.
if RUN_AS_SUBMISSION:
    _run_shell_commands("setup_commands.json", label="setup", check=True)

    # Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
    pythonpath_entries = [
        entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
    ]
    for entry in reversed(pythonpath_entries):
        if entry not in sys.path:
            sys.path.insert(0, entry)
else:
    print("Registration mode: Qwen setup and public benchmark are skipped.")

taaf.kaggle: wrote /usr/local/lib/python3.12/dist-packages/taaf_kaggle_sources.pth (3 source roots)
Registration mode: Qwen setup and public benchmark are skipped.


In [6]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [7]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [8]:
if RUN_AS_SUBMISSION:
    # AGI_8: exact AGI_1 baseline plus one game-agnostic batch safety guard.
    # If a direction leaves the entire visible board unchanged, do not repeat that
    # same direction again inside the current batch. Return control for re-observation.
    import inference.framework.solver as _agi8_solver

    _agi8_original_step_env = _agi8_solver._HarnessGameSession.step_env
    _AGI8_DIRECTIONS = frozenset({"UP", "DOWN", "LEFT", "RIGHT"})


    def _agi8_step_env(self, arguments):
        requested_actions, error = self._normalize_actions(arguments)
        if error is not None or requested_actions is None:
            return self._error_payload(error or "Could not parse action request.")
        if self.should_stop() or _agi8_solver._is_engine_game_over(self.game):
            return self._terminal_payload(requested_actions)

        executed_payloads = []
        total_reward = 0.0
        stop_reason = None
        stop_detail = None
        batch_size = len(requested_actions)
        requested_displays = [
            _agi8_solver._format_action_display(action.id.name, dict(action.data))
            for action in requested_actions
        ]

        for batch_index, action in enumerate(requested_actions, start=1):
            if self.should_stop():
                stop_reason = "stopped"
                break
            if action.id.value not in self.game.current_state.available_actions:
                message = (
                    f"{_agi8_solver._format_action_display(action.id.name, dict(action.data))} "
                    "is not valid right now."
                )
                if executed_payloads:
                    stop_reason = "invalid_action"
                    break
                return self._error_payload(message)

            try:
                payload = self._execute_action(
                    action,
                    batch_index=batch_index,
                    batch_size=batch_size,
                    flush_viewer_payload=False,
                )
            except Exception as exc:
                if executed_payloads:
                    stop_reason = "action_error"
                    break
                return self._error_payload(f"{type(exc).__name__}: {exc}")
            executed_payloads.append(payload)
            total_reward += float(payload.get("reward", 0.0) or 0.0)

            if payload.get("run_complete"):
                stop_reason = "run_complete"
                break
            if payload.get("game_over"):
                stop_reason = "game_over"
                break
            if payload.get("level_completed"):
                stop_reason = "level_completed"
                break

            action_display = requested_displays[batch_index - 1]
            next_display = (
                requested_displays[batch_index] if batch_index < batch_size else None
            )
            if (
                not payload.get("board_changed")
                and action_display in _AGI8_DIRECTIONS
                and next_display == action_display
            ):
                stop_reason = "repeated_no_effect"
                stop_detail = (
                    f"Stopped before repeating {action_display}: the previous identical "
                    "direction left the visible board unchanged. Re-observe before retrying."
                )
                break

        if not executed_payloads:
            return self._error_payload("No action was executed.")

        final_payload = dict(executed_payloads[-1])
        final_payload["reward"] = total_reward
        final_payload["last_reward"] = executed_payloads[-1].get("reward", 0.0)
        final_payload["batched"] = batch_size > 1
        final_payload["requested_count"] = batch_size
        final_payload["executed_count"] = len(executed_payloads)
        final_payload["requested_actions"] = requested_displays
        final_payload["executed_actions"] = [
            str(item.get("action_display") or item.get("action_name") or "")
            for item in executed_payloads
        ]
        final_payload["board_changed"] = any(
            bool(item.get("board_changed")) for item in executed_payloads
        )
        final_payload["stopped_early"] = len(executed_payloads) < batch_size
        if stop_reason is not None:
            final_payload["stop_reason"] = stop_reason
        if stop_detail is not None:
            final_payload["stop_detail"] = stop_detail
        self.write_viewer_payload()
        return final_payload


    _agi8_solver._HarnessGameSession.step_env = _agi8_step_env
    assert _agi8_solver._HarnessGameSession.step_env is _agi8_step_env
    assert bm.solver.concurrency == 28
    assert bm.solver.max_runtime_s_per_game == 7920
    print("AGI_8 patch: repeated-no-effect batch guard active on exact AGI_1 runtime")

    # AGI_9: give inspection-heavy turns one additional model/tool cycle before
    # yielding back to the solver. Analyzer instances are created after this cell.
    import inference.agent.tool_agent as _agi9_tool_agent

    _agi9_original_yield_seconds = (
        _agi9_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS
    )
    assert _agi9_original_yield_seconds == 60.0
    _agi9_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS = 90.0
    assert _agi9_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS == 90.0
    print(
        "AGI_9 scheduling: AGI_8 guard + analyzer yield budget 60s -> 90s"
    )
else:
    print("Registration mode: AGI-8/AGI-9 runtime patches are skipped.")

Registration mode: AGI-8/AGI-9 runtime patches are skipped.


In [9]:
if not run_as_submission:
    # The hidden competition rerun ignores this fast-save artifact.
    import pandas as pd

    submission = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    submission_path = WORKING_DIR / "submission.parquet"
    submission.to_parquet(submission_path, index=False)
    if not submission_path.is_file():
        raise FileNotFoundError(f"Registration artifact was not written: {submission_path}")
    print(f"Wrote Kaggle code-submission registration artifact: {submission_path}")
else:
    run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
    with run_context:
        preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
        print(preamble)
        print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
        print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
        print(f"deploy.kaggle: competition_rerun       = {true_submission}")
        print(f"deploy.kaggle: soft_end_time           = {soft_end}")
        print("---")

        bundled_git_status = BUNDLE_DIR / "git_status.txt"
        if bundled_git_status.is_file():
            (WORKING_DIR / "git_status.txt").write_text(
                bundled_git_status.read_text(encoding="utf-8"),
                encoding="utf-8",
            )

        if true_submission:
            # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
            os.environ.setdefault("ARC_API_KEY", "test-key-123")
            os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
            os.environ.setdefault("SCHEME", "http")
            os.environ.setdefault("HOST", "gateway")
            os.environ.setdefault("PORT", "8001")
            os.environ.setdefault("OPERATION_MODE", "competition")
            os.environ.setdefault("ENVIRONMENTS_DIR", "")
            os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

            deadline = time.monotonic() + 600.0
            last_error = ""
            while time.monotonic() < deadline:
                try:
                    with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                        if response.status < 500:
                            break
                except Exception as exc:
                    last_error = repr(exc)
                time.sleep(5)
            else:
                raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

            bm.games = _competition_games()
            bm.n_passes = 1
            bm.game_weights = None

        try:
            await bm.run(
                soft_end_time=soft_end,
                runtime_environment=target,
                minimal_diagnostics=run_as_submission,
            )
            if not true_submission and Path("/kaggle/input").exists():
                try:
                    import pandas as pd

                    submission = pd.DataFrame(
                        data=[["1_0", "1", True, 1]],
                        columns=["row_id", "game_id", "end_of_game", "score"],
                    )
                    submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
                except Exception as exc:
                    print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
        finally:
            _run_shell_commands("teardown_commands.json", label="teardown", check=False)

Wrote Kaggle code-submission registration artifact: /kaggle/working/submission.parquet
